In [49]:
import tensorflow as tf
import pandas as pd
import numpy as np
data = pd.read_csv("data.csv")
data = data[['R','G', 'B', 'L_cal', 'a_cal', 'b_cal', 'L*', 'a*', 'b*', 'VDO', 'File_Name', 'Crop_Index']]

# 1. normalize 'R', 'G', 'B' (min = 0, max = 255) to [0,1]
data[['R', 'G', 'B']] = data[['R', 'G', 'B']] / 255.0

# 2. normalize 'L_cal', 'L*' (min = 0, max = 100) to [0,1]
data[['L_cal', 'L*']] = data[['L_cal', 'L*']] / 100.0

# 3. normalize 'a_cal', 'b_cal', 'a*', 'b*' (min = -120, max = 120) to [0,1]
# สูตร Min-Max Scaling: (X - Min) / (Max - Min)
# ดังนั้น: (X - (-120)) / (120 - (-120)) = (X + 120) / 240
ab_cols = ['a_cal', 'b_cal', 'a*', 'b*']
data[ab_cols] = (data[ab_cols] + 120.0) / 240.0

# ตรวจสอบผลลัพธ์
print(data.describe()) # ดูค่า min, max ของแต่ละคอลัมน์เพื่อความชัวร์

                  R             G             B         L_cal         a_cal  \
count  20180.000000  20180.000000  20180.000000  20180.000000  20180.000000   
mean       0.492161      0.452944      0.508681      0.568229      0.552066   
std        0.328350      0.288423      0.303449      0.218957      0.133228   
min        0.052215      0.000000      0.058490      0.119400      0.279458   
25%        0.131113      0.154183      0.134449      0.394800      0.493990   
50%        0.622820      0.514110      0.686088      0.593600      0.521042   
75%        0.829643      0.684868      0.769876      0.747100      0.674167   
max        0.860598      0.874538      0.884837      0.894200      0.773458   

              b_cal            L*            a*            b*    Crop_Index  
count  20180.000000  20180.000000  20180.000000  20180.000000  20180.000000  
mean       0.492165      0.520746      0.519823      0.515400      5.500000  
std        0.170485      0.218260      0.091015      0

In [50]:
print(data.describe())

                  R             G             B         L_cal         a_cal  \
count  20180.000000  20180.000000  20180.000000  20180.000000  20180.000000   
mean       0.492161      0.452944      0.508681      0.568229      0.552066   
std        0.328350      0.288423      0.303449      0.218957      0.133228   
min        0.052215      0.000000      0.058490      0.119400      0.279458   
25%        0.131113      0.154183      0.134449      0.394800      0.493990   
50%        0.622820      0.514110      0.686088      0.593600      0.521042   
75%        0.829643      0.684868      0.769876      0.747100      0.674167   
max        0.860598      0.874538      0.884837      0.894200      0.773458   

              b_cal            L*            a*            b*    Crop_Index  
count  20180.000000  20180.000000  20180.000000  20180.000000  20180.000000  
mean       0.492165      0.520746      0.519823      0.515400      5.500000  
std        0.170485      0.218260      0.091015      0

In [51]:
print(data)

              R         G         B   L_cal     a_cal     b_cal      L*  \
0      0.705789  0.004018  0.102107  0.3936  0.766833  0.641625  0.3453   
1      0.718703  0.088197  0.199028  0.4249  0.749083  0.595125  0.3453   
2      0.709311  0.000196  0.097432  0.3942  0.768792  0.645125  0.3453   
3      0.710177  0.000397  0.101993  0.3948  0.769083  0.642500  0.3453   
4      0.718514  0.001622  0.088244  0.3989  0.770208  0.653583  0.3453   
...         ...       ...       ...     ...       ...       ...     ...   
20175  0.831616  0.843380  0.863034  0.8743  0.499667  0.489500  0.9065   
20176  0.847059  0.858824  0.878431  0.8868  0.499625  0.489542  0.9065   
20177  0.841870  0.861851  0.877928  0.8877  0.496417  0.490292  0.9065   
20178  0.846530  0.858362  0.877868  0.8864  0.499583  0.489583  0.9065   
20179  0.847059  0.858824  0.878431  0.8868  0.499625  0.489542  0.9065   

             a*        b*      VDO                 File_Name  Crop_Index  
0      0.631375  0.56754

In [52]:
# สร้างคอลัมน์ 'Base_Color' โดยตัดตัวเลข _1, _2, _3 ด้านหลังออก
# เช่น 'M_136_1' จะกลายเป็น 'M_136'
data['Base_Color'] = data['VDO'].str.rsplit('_', n=1).str[0]


# แบ่ง train test โดย data ที่ column Base_Color มีค่าเดียวกันอยู่ กลุ่มเดียวกัน
# 1. ดึงรายชื่อ Base_Color ทั้งหมดที่ไม่ซ้ำกัน
unique_vdos = data['Base_Color'].unique()

# 2. สับเปลี่ยนลำดับ (Shuffle) รายชื่อ Base_Color เพื่อความสุ่ม
np.random.seed(42)
np.random.shuffle(unique_vdos)

# 3. กำหนดจุดตัดแบ่งข้อมูล (เช่น Train 80%, Test 20%)
split_index = int(len(unique_vdos) * 0.8)

# 4. แบ่งรายชื่อ VDO ออกเป็น 2 กลุ่ม
train_vdo_names = unique_vdos[:split_index]
test_vdo_names = unique_vdos[split_index:]

# 5. กรองข้อมูลจาก DataFrame เดิม
first_data = data[data['Base_Color'].isin(train_vdo_names)].copy()
sec_data = data[data['Base_Color'].isin(test_vdo_names)].copy()

print(f"first_data set: มี {len(first_data)} rows จาก {len(train_vdo_names)} Base_Color")
print(first_data['VDO'].unique())

print(f"sec_data set: มี {len(sec_data)} rows จาก {len(test_vdo_names)} Base_Color")
print(sec_data['VDO'].unique())

first_data set: มี 15320 rows จาก 16 Base_Color
['M_136_1' 'M_136_2' 'M_136_3' 'M_128_1' 'M_128_2' 'M_128_3' 'M_235_1'
 'M_235_2' 'M_235_3' 'M_158_1' 'M_158_2' 'M_158_3' 'M_264_1' 'M_264_2'
 'M_264_3' 'M_266_1' 'M_266_2' 'M_266_3' 'M_327E_1' 'M_327E_2' 'M_327E_3'
 'M_3336_1' 'M_3336_2' 'M_3336_3' 'M_841_1' 'M_841_2' 'M_841_3' 'M_348_1'
 'M_348_2' 'M_348_3' 'M_347_1' 'M_347_2' 'M_347_3' 'M_801_1' 'M_801_2'
 'M_801_3' 'M_814_1' 'M_814_2' 'M_814_3' 'M_504_2' 'M_504_3' 'M_504_1'
 'M_402_1' 'M_402_2' 'M_402_3' 'M_444_1' 'M_444_2' 'M_444_3']
sec_data set: มี 4860 rows จาก 5 Base_Color
['M_324_1' 'M_324_2' 'M_324_3' 'M_327_1' 'M_327_2' 'M_327_3' 'M_835_1'
 'M_835_2' 'M_835_3' 'M_104_1' 'M_104_2' 'M_104_3' 'M_502_1' 'M_502_2'
 'M_502_3']


In [53]:
train_labels = first_data[['L*','a*','b*']]
train_data = first_data.drop(columns=['L*','a*','b*','VDO','File_Name','Crop_Index','Base_Color'])
# สร้าง columns R*G, R*B, G*B, R**2, G**2, B**2
train_data['R*G'] = train_data['R'] * train_data['G']
train_data['R*B'] = train_data['R'] * train_data['B']
train_data['G*B'] = train_data['G'] * train_data['B']

train_data['R**2'] = train_data['R'] ** 2
train_data['G**2'] = train_data['G'] ** 2
train_data['B**2'] = train_data['B'] ** 2

# ตรวจสอบผลลัพธ์
print(train_data.head())

          R         G         B   L_cal     a_cal     b_cal       R*G  \
0  0.705789  0.004018  0.102107  0.3936  0.766833  0.641625  0.002836   
1  0.718703  0.088197  0.199028  0.4249  0.749083  0.595125  0.063388   
2  0.709311  0.000196  0.097432  0.3942  0.768792  0.645125  0.000139   
3  0.710177  0.000397  0.101993  0.3948  0.769083  0.642500  0.000282   
4  0.718514  0.001622  0.088244  0.3989  0.770208  0.653583  0.001166   

        R*B       G*B      R**2          G**2      B**2  
0  0.072066  0.000410  0.498138  1.614464e-05  0.010426  
1  0.143042  0.017554  0.516534  7.778756e-03  0.039612  
2  0.069110  0.000019  0.503122  3.829312e-08  0.009493  
3  0.072433  0.000040  0.504352  1.575000e-07  0.010403  
4  0.063404  0.000143  0.516262  2.632029e-06  0.007787  


In [54]:
test_labels = sec_data[['L*','a*','b*']]
test_data = sec_data.drop(columns=['L*','a*','b*','VDO','File_Name','Crop_Index','Base_Color'])
# สร้าง columns R*G, R*B, G*B, R**2, G**2, B**2
test_data['R*G'] = test_data['R'] * test_data['G']
test_data['R*B'] = test_data['R'] * test_data['B']
test_data['G*B'] = test_data['G'] * test_data['B']

test_data['R**2'] = test_data['R'] ** 2
test_data['G**2'] = test_data['G'] ** 2
test_data['B**2'] = test_data['B'] ** 2

# ตรวจสอบผลลัพธ์
print(test_data.head())

             R         G         B   L_cal     a_cal     b_cal       R*G  \
5860  0.098281  0.297866  0.786597  0.4180  0.607250  0.231667  0.029275   
5861  0.092327  0.296452  0.784768  0.4163  0.606958  0.231500  0.027371   
5862  0.106929  0.299854  0.785563  0.4198  0.606458  0.233500  0.032063   
5863  0.107233  0.303300  0.791643  0.4232  0.606417  0.232375  0.032524   
5864  0.098185  0.290468  0.792824  0.4143  0.614417  0.225708  0.028520   

           R*B       G*B      R**2      G**2      B**2  
5860  0.077308  0.234300  0.009659  0.088724  0.618735  
5861  0.072455  0.232646  0.008524  0.087884  0.615861  
5862  0.083999  0.235554  0.011434  0.089912  0.617109  
5863  0.084890  0.240105  0.011499  0.091991  0.626698  
5864  0.077844  0.230290  0.009640  0.084372  0.628569  


In [55]:
train_data.shape[1]

12

In [56]:
print(len(train_data))
print(len(test_data))

15320
4860


In [57]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# เลือกเฉพาะคอลัมน์ที่เป็น Feature ของ Quadratic Model (รวม 12 ตัว) ['R', 'G', 'B', 'R*G', 'R*B', 'G*B', 'R**2', 'G**2', 'B**2', 'L_cal', 'a_cal', 'b_cal']

X_train = train_data
y_train = train_labels

X_test = test_data
y_test = test_labels

# สร้างโครงสร้าง Neural Network ตาม Paper
model = Sequential([
    # Input layer รับค่าจาก 12 Quadratic Features
    Input(shape=(X_train.shape[1],)),

    # Hidden layer: ใช้ 1 ชั้นและ 8 นิวรอนตามที่ระบุในเปเปอร์
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),

    # Output layer: 3 นิวรอน สำหรับค่า L*, a*, b* (ใช้ linear activation เพราะเป็น Regression)
    Dense(3, activation='linear')
])

# คอมไพล์โมเดล โดยใช้ loss เป็น Mean Absolute Error (MAE) ตามสมการในเปเปอร์
model.compile(optimizer=Adam(learning_rate=0.001),loss='mae',metrics=['mae', 'mse'])

# สรุปโครงสร้างโมเดล
model.summary()

# ตั้งค่า Early Stopping เพื่อหยุดเทรนเมื่อ Validation Loss ไม่ลดลง (ตามเปเปอร์)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=40, # หาก error บน validation set ไม่ลดลงติดต่อกัน 40 epochs ให้หยุด
    restore_best_weights=True
)

# เริ่มการเทรนโมเดล
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=500, # ตั้งเผื่อไว้ Early Stopping จะหยุดให้เอง
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# ประเมินผลลัพธ์กับ Test Set
test_loss, test_mae, test_mse = model.evaluate(X_test, y_test)
print(f"Test MAE: {test_mae:.4f}")

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 32)             │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 995 (3.89 KB)

 Trainable params: 995 (3.89 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
479/479 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0385 - mae: 0.0385 - mse: 0.0090 - val_loss: 0.0182 - val_mae: 0.0182 - val_mse: 6.2919e-04
Epoch 2/500
479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 932us/step - loss: 0.0081 - mae: 0.0081 - mse: 2.0256e-04 - val_loss: 0.0176 - val_mae: 0.0176 - val_mse: 5.8179e-04
Epoch 3/500
479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 916us/step - loss: 0.0072 - mae: 0.0072 - mse: 1.5918e-04 - val_loss: 0.0170 - val_mae: 0.0170 - val_mse: 5.3589e-04
Epoch 4/500
479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 912us/step - loss: 0.0067 - mae: 0.0067 - mse: 1.3513e-04 - val_loss: 0.0165 - val_mae: 0.0165 - val_mse: 4.7937e-04
Epoch 5/500
479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 912us/step - loss: 0.0063 - mae: 0.0063 - mse: 1.1982e-04 - val_loss: 0.0158 - val_mae: 0.0158 - val_mse: 4.0616e-04
Epoch 6/500
479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 914us/step - loss: 0.0057 - mae: 0.0057 - mse: 1.0502e-04 - val_loss: 0.0148 - val_mae: 0.0148 - val_mse: 3.5183e-04
Epoch 7/500
479/479 ━━━━━━━━━━━━━━━━━━━━

In [60]:
import numpy as np
model = tf.keras.models.load_model('my_model.keras')
# 1. ทำนายผลลัพธ์จาก Test set ทั้งหมดในรวดเดียว (ไม่ต้องใช้ loop)
preds_norm = model.predict(X_test)

# 2. Denormalize ผลทำนายและผลจริง กลับเป็นสเกลปกติ
# L* สเกลเดิมคือ 0 ถึง 100 (ตอนแปลงเราหาร 100)
preds_L = preds_norm[:, 0] * 100.0
trues_L = y_test['L*'].values * 100.0

# a* และ b* สเกลเดิมคือ -120 ถึง 120 (ตอนแปลงเราทำ (x+120)/240)
preds_a = (preds_norm[:, 1] * 240.0) - 120.0
trues_a = (y_test['a*'].values * 240.0) - 120.0

preds_b = (preds_norm[:, 2] * 240.0) - 120.0
trues_b = (y_test['b*'].values * 240.0) - 120.0

# นำกลับมารวมเป็น Array เดียวกัน
preds_all = np.column_stack((preds_L, preds_a, preds_b))
trues_all = np.column_stack((trues_L, trues_a, trues_b))

# 3. คำนวณค่า RMSE
rmse_L = np.sqrt(((preds_all[:,0] - trues_all[:,0])**2).mean())
rmse_a = np.sqrt(((preds_all[:,1] - trues_all[:,1])**2).mean())
rmse_b = np.sqrt(((preds_all[:,2] - trues_all[:,2])**2).mean())
rmse_total = np.sqrt(((preds_all - trues_all)**2).mean())

# 4. คำนวณ Error (Mean Normalized Error) ตามสมการใน Paper
e_L = np.abs(preds_all[:,0] - trues_all[:,0]).mean() / 100.0
e_a = np.abs(preds_all[:,1] - trues_all[:,1]).mean() / 240.0
e_b = np.abs(preds_all[:,2] - trues_all[:,2]).mean() / 240.0

# คำนวณ Total Error เป็นเปอร์เซ็นต์
e_total = ((e_L + e_a + e_b) / 3.0) * 100.0

# 5. แสดงผลลัพธ์เปรียบเทียบ
print(f"\nNN(Quad+Lab) RMSE  L:{rmse_L:.2f}  a:{rmse_a:.2f}  b:{rmse_b:.2f}  total:{rmse_total:.2f}")
print(f"NN(Quad+Lab) Error (paper style): {e_total:.2f}%\n")

print("--- เทียบกับผลลัพธ์อ้างอิง ---")
print(f"Paper Quadratic (RGB only):         1.23%") # อ้างอิงจาก Table 3 ใน Paper
print(f"Paper NN (RGB only):                0.93%") # อ้างอิงจาก Table 3 ใน Paper

152/152 ━━━━━━━━━━━━━━━━━━━━ 0s 490us/step

NN(Quad+Lab) RMSE  L:1.57  a:2.06  b:2.90  total:2.24
NN(Quad+Lab) Error (paper style): 0.98%

--- เทียบกับผลลัพธ์อ้างอิง ---
Paper Quadratic (RGB only):         1.23%
Paper NN (RGB only):                0.93%


In [59]:
# Save Model Tensor
# Save the entire model 
model.save('my_model.keras')
# Load the model back later
loaded_model = tf.keras.models.load_model('my_model.keras')


32-16 : 1.09%